# CSO Employment Data Preprocessing

> Notebook version: **2.0 — separate label frequency tables**

This notebook loads and preprocesses the five CSO PxStat CSV datasets used by LaborLens. Each dataset follows the same workflow:

1. Load the original CSV with UTF-8 BOM handling.
2. Trim column names and text values.
3. Convert `VALUE` to a nullable numeric column.
4. Derive `YEAR` from the source period and retain 2019–2025 only.
5. Remove exact duplicate rows.
6. Convert descriptive dimensions to pandas `category` dtype.
7. Display column definitions, a sample, and categorical label counts.

Empty statistical values remain missing; they are never converted to zero.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_VERSION = "2.0-separate-label-frequency-tables"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

START_YEAR = 2019
END_YEAR = 2025
SAMPLE_ROWS = 5

# Support execution from either the repository root or this notebook's directory.
candidate_directories = [Path.cwd(), Path.cwd() / "data" / "cso"]
DATA_DIR = next(
    directory
    for directory in candidate_directories
    if (directory / "ALF01.csv").exists()
)

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"pandas version: {pd.__version__}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Analysis period: {START_YEAR}–{END_YEAR}")

Notebook version: 2.0-separate-label-frequency-tables
pandas version: 2.3.3
Data directory: /Users/hazely/Projects/GithubLocal/LaborLens/data/cso
Analysis period: 2019–2025


## Dataset configuration and reusable preprocessing

The configuration below documents every source column. Columns whose names appear in `category_columns` are treated as categorical labels. Source code columns are retained as strings so that the original CSO identifiers are never lost.

In [2]:
DATASETS = {
    "ALF01": {
        "title": "Annual Employment Rate",
        "period_column": "Year",
        "category_columns": [
            "Statistic Label", "Age Group", "Sex",
            "Education Attainment Level", "NUTS 2 Region", "UNIT",
        ],
        "column_meanings": {
            "STATISTIC": "CSO code identifying the statistical measure.",
            "Statistic Label": "Readable name of the statistical measure.",
            "TLIST(A1)": "CSO annual time code.",
            "Year": "Calendar year represented by the observation.",
            "C02076V02508": "CSO code for the age-group dimension.",
            "Age Group": "Readable age range or aggregate age category.",
            "C02199V02655": "CSO code for the sex dimension.",
            "Sex": "Sex category used by the source table.",
            "C04283V05060": "CSO code for education attainment level.",
            "Education Attainment Level": "Highest education-attainment category.",
            "C03788V04538": "CSO code for the NUTS 2 regional dimension.",
            "NUTS 2 Region": "Irish NUTS 2 region or national aggregate.",
            "UNIT": "Measurement unit, normally a percentage for this table.",
            "VALUE": "Published observation value; blank means missing, not zero.",
            "YEAR": "Derived calendar year used for the 2019–2025 filter.",
        },
    },
    "MUM01": {
        "title": "Seasonally Adjusted Monthly Unemployment",
        "period_column": "Month",
        "category_columns": ["Statistic Label", "Age Group", "Sex", "UNIT"],
        "column_meanings": {
            "STATISTIC": "CSO code identifying unemployment count or unemployment rate.",
            "Statistic Label": "Readable name of the unemployment measure.",
            "TLIST(M1)": "CSO monthly time code in YYYYMM form.",
            "Month": "Readable observation month.",
            "C02076V02508": "CSO code for the age-group dimension.",
            "Age Group": "Readable age range or aggregate age category.",
            "C02199V02655": "CSO code for the sex dimension.",
            "Sex": "Sex category used by the source table.",
            "UNIT": "Measurement unit, such as Thousand or %.",
            "VALUE": "Published seasonally adjusted value; blank means missing.",
            "YEAR": "Derived calendar year used for the 2019–2025 filter.",
        },
    },
    "QLF50": {
        "title": "Quarterly Employment Rate",
        "period_column": "Quarter",
        "category_columns": [
            "Statistic Label", "Sex", "Education Attainment Level",
            "Age Group", "UNIT",
        ],
        "column_meanings": {
            "STATISTIC": "CSO code identifying the quarterly employment-rate measure.",
            "Statistic Label": "Readable name of the statistical measure.",
            "TLIST(Q1)": "CSO quarterly time code.",
            "Quarter": "Calendar quarter in YYYYQn form.",
            "C02199V02655": "CSO code for the sex dimension.",
            "Sex": "Sex category used by the source table.",
            "C04283V05060": "CSO code for education attainment level.",
            "Education Attainment Level": "Highest education-attainment category.",
            "C02076V02508": "CSO code for the age-group dimension.",
            "Age Group": "Readable age range or aggregate age category.",
            "UNIT": "Measurement unit, normally a percentage for this table.",
            "VALUE": "Published quarterly employment-rate value; blank means missing.",
            "YEAR": "Derived calendar year used for the 2019–2025 filter.",
        },
    },
    "QLF59": {
        "title": "Persons Aged 15 Years and Over in Employment",
        "period_column": "Quarter",
        "category_columns": [
            "Statistic Label", "Citizenship",
            "NACE Rev 2.1 Economic Sector", "UNIT",
        ],
        "column_meanings": {
            "STATISTIC": "CSO code identifying the employment-count measure.",
            "Statistic Label": "Readable name of the employment measure.",
            "TLIST(Q1)": "CSO quarterly time code.",
            "Quarter": "Calendar quarter in YYYYQn form.",
            "C04004V04762": "CSO code for the citizenship dimension.",
            "Citizenship": "Citizenship group represented by the observation.",
            "C04498V05282": "CSO code for the NACE Rev. 2.1 sector dimension.",
            "NACE Rev 2.1 Economic Sector": "Economic sector under NACE Rev. 2.1.",
            "UNIT": "Measurement unit, normally Thousand.",
            "VALUE": "Published employment count; blank means missing.",
            "YEAR": "Derived calendar year used for the 2019–2025 filter.",
        },
    },
    "EHQ03": {
        "title": "Earnings, Hours, Employment and Labour Costs",
        "period_column": "Quarter",
        "category_columns": [
            "Statistic Label", "Economic Sector NACE Rev 2",
            "Type of Employee", "UNIT",
        ],
        "column_meanings": {
            "STATISTIC": "CSO code identifying the earnings, hours, employment, or labour-cost measure.",
            "Statistic Label": "Readable name of the statistical measure.",
            "TLIST(Q1)": "CSO quarterly time code.",
            "Quarter": "Calendar quarter in YYYYQn form.",
            "C02665V03225": "CSO code for the NACE Rev. 2 economic-sector dimension.",
            "Economic Sector NACE Rev 2": "Economic sector under NACE Rev. 2.",
            "C02397V02888": "CSO code for the employee-type dimension.",
            "Type of Employee": "Employee occupational or aggregate category.",
            "UNIT": "Measurement unit, such as Euro, Hours, Number, or %.",
            "VALUE": "Published observation value; blank means missing.",
            "YEAR": "Derived calendar year used for the 2019–2025 filter.",
        },
    },
}


def preprocess_dataset(dataset_id):
    """Load one CSO CSV and apply the shared deterministic cleaning steps."""
    config = DATASETS[dataset_id]
    path = DATA_DIR / f"{dataset_id}.csv"

    # utf-8-sig removes the optional byte-order mark visible in CSO CSV headers.
    frame = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    frame.columns = frame.columns.str.strip()

    # Remove surrounding whitespace without changing CSO codes or labels.
    text_columns = frame.select_dtypes(include=["object", "string"]).columns
    frame[text_columns] = frame[text_columns].apply(lambda column: column.str.strip())

    # Blank published values are missing observations, never numeric zero.
    frame["VALUE"] = pd.to_numeric(
        frame["VALUE"].replace("", pd.NA).str.replace(",", "", regex=False),
        errors="coerce",
    ).astype("Float64")

    # All selected source periods begin with a four-digit calendar year.
    frame["YEAR"] = pd.to_numeric(
        frame[config["period_column"]].str.extract(r"^(\d{4})", expand=False),
        errors="coerce",
    ).astype("Int64")

    source_rows = len(frame)
    frame = frame.loc[frame["YEAR"].between(START_YEAR, END_YEAR)].copy()
    in_scope_rows = len(frame)
    duplicate_rows = int(frame.duplicated().sum())
    frame = frame.drop_duplicates().reset_index(drop=True)

    for column in config["category_columns"]:
        frame[column] = frame[column].replace("", pd.NA).astype("category")

    metrics = {
        "dataset": dataset_id,
        "source_rows": source_rows,
        "rows_in_2019_2025": in_scope_rows,
        "exact_duplicates_removed": duplicate_rows,
        "final_rows": len(frame),
        "missing_values": int(frame["VALUE"].isna().sum()),
        "minimum_year": int(frame["YEAR"].min()),
        "maximum_year": int(frame["YEAR"].max()),
    }
    return frame, metrics


def column_glossary(dataset_id):
    """Return the documented meaning and processed dtype of every column."""
    config = DATASETS[dataset_id]
    frame, _ = PROCESSED[dataset_id]
    return pd.DataFrame(
        [
            {
                "column": column,
                "meaning": meaning,
                "processed_dtype": str(frame[column].dtype),
            }
            for column, meaning in config["column_meanings"].items()
        ]
    )


def category_frequency_table(dataset_id, column):
    """Return every label's frequency and percentage of all processed rows."""
    frame, _ = PROCESSED[dataset_id]
    total_rows = len(frame)
    frequencies = (
        frame[column]
        .value_counts(dropna=False)
        .rename_axis("label")
        .reset_index(name="row_count")
    )
    frequencies["label"] = frequencies["label"].astype("string").fillna("<MISSING>")
    frequencies["percentage_of_rows"] = (
        frequencies["row_count"].div(total_rows).mul(100).round(2)
    )
    return frequencies


def display_dataset(dataset_id):
    """Display the common documentation, sample, and category summary workflow."""
    config = DATASETS[dataset_id]
    frame, metrics = PROCESSED[dataset_id]
    display(Markdown(f"### {dataset_id} — {config['title']}"))
    display(Markdown("**Preprocessing summary**"))
    display(pd.DataFrame([metrics]))
    display(Markdown("**Column meanings**"))
    display(column_glossary(dataset_id))
    display(Markdown(f"**First {SAMPLE_ROWS} processed rows**"))
    display(frame.head(SAMPLE_ROWS))
    display(Markdown("**Categorical distributions**"))
    for column in config["category_columns"]:
        label_count = int(frame[column].nunique(dropna=True))
        display(Markdown(f"#### `{column}` — {label_count} labels"))
        display(category_frequency_table(dataset_id, column))


# Load all files with exactly the same preprocessing function.
PROCESSED = {dataset_id: preprocess_dataset(dataset_id) for dataset_id in DATASETS}
print("Loaded:", ", ".join(PROCESSED))

Loaded: ALF01, MUM01, QLF50, QLF59, EHQ03


## 1. ALF01 — Annual Employment Rate

Annual employment rates provide a compact view of the entire 2019–2025 period, including age, sex, education, and regional dimensions.

In [3]:
display_dataset("ALF01")

### ALF01 — Annual Employment Rate

**Preprocessing summary**

,dataset,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
0,ALF01,5040,5040,0,5040,2003,2019,2025


**Column meanings**

,column,meaning,processed_dtype
0,STATISTIC,CSO code identifying the statistical measure.,object
1,Statistic Label,Readable name of the statistical measure.,category
2,TLIST(A1),CSO annual time code.,object
3,Year,Calendar year represented by the observation.,object
4,C02076V02508,CSO code for the age-group dimension.,object
5,Age Group,Readable age range or aggregate age category.,category
6,C02199V02655,CSO code for the sex dimension.,object
7,Sex,Sex category used by the source table.,category
8,C04283V05060,CSO code for education attainment level.,object
9,Education Attainment Level,Highest education-attainment category.,category


...

,label,row_count,percentage_of_rows
0,Less than primary (Level 0),840,16.67
1,Level of education - not stated,840,16.67
2,Levels of Education (Levels 0-8),840,16.67
3,Primary and lower secondary education (Levels 1-2),840,16.67
4,Tertiary education (Levels 5-8),840,16.67
5,Upper secondary and post-secondary non-tertiary education (Levels 3 and 4),840,16.67


#### `NUTS 2 Region` — 4 labels

,label,row_count,percentage_of_rows
0,Eastern and Midland,1260,25.0
1,Ireland,1260,25.0
2,Northern and Western,1260,25.0
3,Southern,1260,25.0


#### `UNIT` — 1 labels

,label,row_count,percentage_of_rows
0,%,5040,100.0


## 2. MUM01 — Seasonally Adjusted Monthly Unemployment

This table contains monthly unemployment counts and rates. The statistical label and unit must be retained together so that counts and percentages are not mixed.

In [4]:
display_dataset("MUM01")

### MUM01 — Seasonally Adjusted Monthly Unemployment

**Preprocessing summary**

,dataset,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
0,MUM01,6192,1512,0,1512,0,2019,2025


**Column meanings**

,column,meaning,processed_dtype
0,STATISTIC,CSO code identifying unemployment count or unemployment rate.,object
1,Statistic Label,Readable name of the unemployment measure.,category
2,TLIST(M1),CSO monthly time code in YYYYMM form.,object
3,Month,Readable observation month.,object
4,C02076V02508,CSO code for the age-group dimension.,object
5,Age Group,Readable age range or aggregate age category.,category
6,C02199V02655,CSO code for the sex dimension.,object
7,Sex,Sex category used by the source table.,category
8,UNIT,"Measurement unit, such as Thousand or %.",category
9,VALUE,Published seasonally adjusted value; blank means missing.,Float64


...

,label,row_count,percentage_of_rows
0,15 - 24 years,504,33.33
1,15 - 74 years,504,33.33
2,25 - 74 years,504,33.33


#### `Sex` — 3 labels

,label,row_count,percentage_of_rows
0,Both sexes,504,33.33
1,Female,504,33.33
2,Male,504,33.33


#### `UNIT` — 2 labels

,label,row_count,percentage_of_rows
0,%,756,50.0
1,Thousand,756,50.0


## 3. QLF50 — Quarterly Employment Rate

Quarterly employment rates add more temporal detail than ALF01 while retaining sex, education, and age dimensions.

In [5]:
display_dataset("QLF50")

### QLF50 — Quarterly Employment Rate

**Preprocessing summary**

,dataset,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
0,QLF50,5940,5544,0,5544,1830,2019,2025


**Column meanings**

,column,meaning,processed_dtype
0,STATISTIC,CSO code identifying the quarterly employment-rate measure.,object
1,Statistic Label,Readable name of the statistical measure.,category
2,TLIST(Q1),CSO quarterly time code.,object
3,Quarter,Calendar quarter in YYYYQn form.,object
4,C02199V02655,CSO code for the sex dimension.,object
5,Sex,Sex category used by the source table.,category
6,C04283V05060,CSO code for education attainment level.,object
7,Education Attainment Level,Highest education-attainment category.,category
8,C02076V02508,CSO code for the age-group dimension.,object
9,Age Group,Readable age range or aggregate age category.,category


...

,label,row_count,percentage_of_rows
0,Less than primary (Level 0),924,16.67
1,Level of education - not stated,924,16.67
2,Levels of Education (Levels 0-8),924,16.67
3,Primary and lower secondary education (Levels 1-2),924,16.67
4,Tertiary education (Levels 5-8),924,16.67
5,Upper secondary and post-secondary non-tertiary education (Levels 3 and 4),924,16.67


#### `Age Group` — 11 labels

,label,row_count,percentage_of_rows
0,20 - 24 years,504,9.09
1,25 - 29 years,504,9.09
2,25 - 54 years,504,9.09
3,30 - 34 years,504,9.09
4,35 - 39 years,504,9.09
5,40 - 44 years,504,9.09
6,45 - 49 years,504,9.09
7,50 - 54 years,504,9.09
8,55 - 59 years,504,9.09
9,60 - 64 years,504,9.09


#### `UNIT` — 1 labels

,label,row_count,percentage_of_rows
0,%,5544,100.0


## 4. QLF59 — Persons Aged 15 Years and Over in Employment

This table provides quarterly employment counts by citizenship and NACE Rev. 2.1 economic sector.

In [6]:
display_dataset("QLF59")

### QLF59 — Persons Aged 15 Years and Over in Employment

**Preprocessing summary**

,dataset,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
0,QLF59,3780,3528,0,3528,1914,2019,2025


**Column meanings**

,column,meaning,processed_dtype
0,STATISTIC,CSO code identifying the employment-count measure.,object
1,Statistic Label,Readable name of the employment measure.,category
2,TLIST(Q1),CSO quarterly time code.,object
3,Quarter,Calendar quarter in YYYYQn form.,object
4,C04004V04762,CSO code for the citizenship dimension.,object
5,Citizenship,Citizenship group represented by the observation.,category
6,C04498V05282,CSO code for the NACE Rev. 2.1 sector dimension.,object
7,NACE Rev 2.1 Economic Sector,Economic sector under NACE Rev. 2.1.,category
8,UNIT,"Measurement unit, normally Thousand.",category
9,VALUE,Published employment count; blank means missing.,Float64


...

,label,row_count,percentage_of_rows
0,All Countries,504,14.29
1,All countries excluding Ireland,504,14.29
2,"All countries excluding Ireland,United Kingdom and EU272020",504,14.29
3,EU14 excl Irl (countries in the EU pre 2004 excluding UK & Ireland),504,14.29
4,EU15 to EU27 (accession countries joined post 2004),504,14.29
5,Ireland,504,14.29
6,United Kingdom,504,14.29


#### `NACE Rev 2.1 Economic Sector` — 18 labels

,label,row_count,percentage_of_rows
0,Accommodation and Food Service Activities (I),196,5.56
1,Administrative and Support Service Activities (O),196,5.56
2,Transportation and Storage (H),196,5.56
3,Services (G-V),196,5.56
4,Public Administration and Defence; Compulsory Social Security (P),196,5.56
5,"Professional, Scientific and Technical Activities (N)",196,5.56
6,Other NACE Activities (S to V),196,5.56
7,NACE Unknown,196,5.56
8,"Information and Communication (J,K)",196,5.56
9,Industry and Construction (B-F),196,5.56


#### `UNIT` — 1 labels

,label,row_count,percentage_of_rows
0,Thousand,3528,100.0


## 5. EHQ03 — Earnings, Hours, Employment and Labour Costs

EHQ03 is the largest source file. It contains several statistical measures and units, so downstream analysis must always filter by both `STATISTIC` and `UNIT`.

In [7]:
display_dataset("EHQ03")

### EHQ03 — Earnings, Hours, Employment and Labour Costs

**Preprocessing summary**

,dataset,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
0,EHQ03,130536,49392,0,49392,43120,2019,2025


**Column meanings**

,column,meaning,processed_dtype
0,STATISTIC,"CSO code identifying the earnings, hours, employment, or labour-cost measure.",object
1,Statistic Label,Readable name of the statistical measure.,category
2,TLIST(Q1),CSO quarterly time code.,object
3,Quarter,Calendar quarter in YYYYQn form.,object
4,C02665V03225,CSO code for the NACE Rev. 2 economic-sector dimension.,object
5,Economic Sector NACE Rev 2,Economic sector under NACE Rev. 2.,category
6,C02397V02888,CSO code for the employee-type dimension.,object
7,Type of Employee,Employee occupational or aggregate category.,category
8,UNIT,"Measurement unit, such as Euro, Hours, Number, or %.",category
9,VALUE,Published observation value; blank means missing.,Float64


...

,label,row_count,percentage_of_rows
0,Accommodation and food service activities (I),2352,4.76
1,Industry (B to E),2352,4.76
2,Transportation and storage (H),2352,4.76
3,Real estate activities (L),2352,4.76
4,Public administration and defence; compulsory social security (O),2352,4.76
5,"Professional, scientific and technical activities (M)",2352,4.76
6,Other service activities (S),2352,4.76
7,Mining and quarrying (B),2352,4.76
8,Manufacturing (C),2352,4.76
9,Information and communication (J),2352,4.76


#### `Type of Employee` — 4 labels

,label,row_count,percentage_of_rows
0,All employees,12348,25.0
1,"Clerical, sales and service employees",12348,25.0
2,"Managers, professionals and associated professionals",12348,25.0
3,"Production, transport, craft and other manual workers",12348,25.0


#### `UNIT` — 3 labels

,label,row_count,percentage_of_rows
0,Euro,42336,85.71
1,Hours,4704,9.52
2,Number,2352,4.76


## Combined preprocessing summary

This final table makes it easy to compare input size, retained records, duplicate removal, and missing statistical values across all five datasets.

In [8]:
combined_summary = pd.DataFrame(
    [metrics for _, metrics in PROCESSED.values()]
).set_index("dataset")
combined_summary

,source_rows,rows_in_2019_2025,exact_duplicates_removed,final_rows,missing_values,minimum_year,maximum_year
dataset,,,,,,,
ALF01,5040,5040,0,5040,2003,2019,2025
MUM01,6192,1512,0,1512,0,2019,2025
QLF50,5940,5544,0,5544,1830,2019,2025
QLF59,3780,3528,0,3528,1914,2019,2025
EHQ03,130536,49392,0,49392,43120,2019,2025
